# Часть 3. Оптимизация классификатора для инференса на CPU
Отправная точка — модель `cointegrated/rubert-tiny2`, дообученная в части 2.

In [ ]:
!pip install -q datasets transformers torch scikit-learn optimum[exporters] onnxruntime

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import onnxruntime as ort

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Загрузка модели и тестовых данных

In [ ]:
CHECKPOINT_DIR = './checkpoints'
MODEL_NAME     = 'cointegrated/rubert-tiny2'
MAX_LEN        = 128
BATCH_SIZE     = 32
LABEL_NAMES    = {0: 'NEUTRAL', 1: 'POSITIVE', 2: 'NEGATIVE'}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if os.path.isdir(CHECKPOINT_DIR):
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
    print('Загружен дообученный чекпоинт')
else:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    print('Чекпоинт не найден — загружена базовая модель (без файнтюнинга)')

model.eval()
model = model.cpu()

In [ ]:
df = load_dataset('MonoHime/ru_sentiment_dataset', split='train').to_pandas()
df = df[['text', 'sentiment']].dropna().rename(columns={'sentiment': 'label'})
df['label'] = df['label'].astype(int)

sample = df.groupby('label', group_keys=False).apply(
    lambda g: g.sample(int(30_000 * len(g) / len(df)), random_state=42)
).reset_index(drop=True)

_, tmp = train_test_split(sample, test_size=0.2, stratify=sample['label'], random_state=42)
_, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp['label'], random_state=42)

print(f'Test: {len(test_df):,} примеров')

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True,
            padding='max_length', max_length=MAX_LEN, return_tensors='pt',
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

test_ds     = SentimentDataset(test_df['text'], test_df['label'])
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
sample_batch = next(iter(test_loader))

## 2. Выбор метода оптимизации

Рассматривались три подхода:

**Dynamic INT8 Quantization** — переводит веса линейных слоёв из float32 в int8 без переобучения и без калибровочных данных. Даёт ~2–4× ускорение на CPU и уменьшает размер модели примерно в 4 раза при минимальной потере качества (<1% F1). Встроен в PyTorch, минимум зависимостей.

**ONNX Runtime** — экспортирует модель в унифицированный формат ONNX и запускает её через оптимизированный рантайм. Добавляет независимость от PyTorch на инференсе и часто даёт лучшее ускорение, чем динамическая квантизация, за счёт graph-оптимизаций (слияние слоёв, удаление избыточных операций).

**Pruning / Knowledge Distillation** — требуют дообучения, то есть существенно больше ресурсов и времени. Оправданы при жёстких требованиях к размеру модели, но выходят за рамки оптимизации «для инференса».

**Выбор: Dynamic Quantization + ONNX Runtime.**  
Оба метода не требуют переобучения. Применяем оба последовательно, чтобы дать сравнительную картину: quantization — простой и нативный путь, ONNX Runtime — более агрессивная оптимизация. Итоговая сводная таблица покажет, какой прирост даёт каждый.

## 3. Оптимизация

### 3.1 Вспомогательные функции

In [ ]:
def get_model_size_mb(model):
    path = '/tmp/_tmp_model.pt'
    torch.save(model.state_dict(), path)
    size = os.path.getsize(path) / 1024 ** 2
    os.remove(path)
    return size


def measure_pytorch(model, batch, n_warmup=5, n_measure=30):
    model.eval()
    inputs = {k: v for k, v in batch.items() if k != 'labels'}

    with torch.no_grad():
        for _ in range(n_warmup):
            model(**inputs)

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_measure):
            model(**inputs)
    elapsed = time.perf_counter() - start

    return elapsed / n_measure * 1000


def get_pytorch_preds(model, loader):
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels')
            out    = model(**batch)
            all_preds.extend(out.logits.argmax(-1).tolist())
            all_labels.extend(labels.tolist())
    return np.array(all_labels), np.array(all_preds)

### 3.2 Базовая модель (fp32)

In [ ]:
base_ms   = measure_pytorch(model, sample_batch)
base_size = get_model_size_mb(model)
y_true, base_preds = get_pytorch_preds(model, test_loader)

base_f1  = f1_score(y_true, base_preds, average='macro')
base_acc = accuracy_score(y_true, base_preds)

print(f'Размер   : {base_size:.1f} МБ')
print(f'Скорость : {base_ms:.1f} мс/батч')
print(f'F1-macro : {base_f1:.4f}')
print(f'Accuracy : {base_acc:.4f}')

### 3.3 Dynamic INT8 Quantization

In [ ]:
quant_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8,
)

quant_ms   = measure_pytorch(quant_model, sample_batch)
quant_size = get_model_size_mb(quant_model)
_, quant_preds = get_pytorch_preds(quant_model, test_loader)

quant_f1  = f1_score(y_true, quant_preds, average='macro')
quant_acc = accuracy_score(y_true, quant_preds)

print(f'Размер   : {quant_size:.1f} МБ  (было {base_size:.1f} МБ)')
print(f'Скорость : {quant_ms:.1f} мс/батч  (было {base_ms:.1f} мс)')
print(f'F1-macro : {quant_f1:.4f}  (было {base_f1:.4f})')
print(f'Accuracy : {quant_acc:.4f}  (было {base_acc:.4f})')

### 3.4 ONNX Runtime

Экспортируем модель в ONNX, затем запускаем через `onnxruntime` с включёнными graph-оптимизациями. Оптимизации второго уровня (`ORT_ENABLE_EXTENDED`) сливают последовательные операции (LayerNorm, Gelu, Attention), что сокращает число ядерных вызовов на CPU.

In [ ]:
ONNX_PATH = '/tmp/rubert_tiny2.onnx'

dummy_input_ids = sample_batch['input_ids']
dummy_attn_mask = sample_batch['attention_mask']

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attn_mask),
    ONNX_PATH,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids':      {0: 'batch', 1: 'seq'},
        'attention_mask': {0: 'batch', 1: 'seq'},
        'logits':         {0: 'batch'},
    },
    opset_version=14,
)
print('Экспорт завершён')

In [ ]:
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
sess_options.intra_op_num_threads = os.cpu_count()

ort_session = ort.InferenceSession(
    ONNX_PATH,
    sess_options=sess_options,
    providers=['CPUExecutionProvider'],
)

print('ONNX Runtime сессия создана')

In [ ]:
def measure_onnx(session, batch, n_warmup=5, n_measure=30):
    inputs = {
        'input_ids':      batch['input_ids'].numpy(),
        'attention_mask': batch['attention_mask'].numpy(),
    }

    for _ in range(n_warmup):
        session.run(None, inputs)

    start = time.perf_counter()
    for _ in range(n_measure):
        session.run(None, inputs)
    return (time.perf_counter() - start) / n_measure * 1000


def get_onnx_preds(session, loader):
    all_preds, all_labels = [], []
    for batch in loader:
        labels = batch['labels'].numpy()
        inputs = {
            'input_ids':      batch['input_ids'].numpy(),
            'attention_mask': batch['attention_mask'].numpy(),
        }
        logits = session.run(None, inputs)[0]
        all_preds.extend(logits.argmax(axis=-1).tolist())
        all_labels.extend(labels.tolist())
    return np.array(all_labels), np.array(all_preds)


onnx_ms   = measure_onnx(ort_session, sample_batch)
onnx_size = os.path.getsize(ONNX_PATH) / 1024 ** 2
_, onnx_preds = get_onnx_preds(ort_session, test_loader)

onnx_f1  = f1_score(y_true, onnx_preds, average='macro')
onnx_acc = accuracy_score(y_true, onnx_preds)

print(f'Размер   : {onnx_size:.1f} МБ')
print(f'Скорость : {onnx_ms:.1f} мс/батч')
print(f'F1-macro : {onnx_f1:.4f}')
print(f'Accuracy : {onnx_acc:.4f}')

## 4. Сравнительная таблица

In [ ]:
comparison = pd.DataFrame({
    'Модель': [
        'Оригинал (fp32)',
        'Dynamic INT8 Quant',
        'ONNX Runtime',
    ],
    'Размер, МБ': [base_size, quant_size, onnx_size],
    'Время батча, мс': [base_ms, quant_ms, onnx_ms],
    'Ускорение': [
        1.0,
        round(base_ms / quant_ms, 2),
        round(base_ms / onnx_ms, 2),
    ],
    'F1-macro': [base_f1, quant_f1, onnx_f1],
    'Accuracy': [base_acc, quant_acc, onnx_acc],
    'Потеря F1, %': [
        0.0,
        round((base_f1 - quant_f1) / base_f1 * 100, 2),
        round((base_f1 - onnx_f1)  / base_f1 * 100, 2),
    ],
}).set_index('Модель')

comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = [
    ('Время батча, мс',  'tab:blue',   'Скорость (мс/батч) — меньше лучше'),
    ('Размер, МБ',       'tab:orange', 'Размер модели (МБ) — меньше лучше'),
    ('F1-macro',         'tab:green',  'F1-macro — больше лучше'),
]

for ax, (col, color, title) in zip(axes, metrics):
    comparison[col].plot(kind='bar', ax=ax, color=color, edgecolor='black', rot=20)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Сравнение методов оптимизации', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()